# 라이브러리

In [1]:
import os
import json
import hashlib

from dotenv import load_dotenv
from llama_index.readers.docling import DoclingReader
from llama_index.core.node_parser import MarkdownNodeParser

# 환경 변수

In [2]:
# load .env
load_dotenv()

GEMINI_API_KEY = os.environ.get('GEMINI_API_KEY')

# PDF 파싱

In [3]:
# VectorDB에서 논문 단위의 namespace를 구성하기 위한 해시값 생성
def make_paper_id_from_file(file_path: str) -> str:
    with open(file_path, "rb") as f:
        file_bytes = f.read()
    return hashlib.sha256(file_bytes).hexdigest()

In [4]:
def load_and_parse_with_docling(file_path):
    paper_id = make_paper_id_from_file(file_path)
    
    reader = DoclingReader()
    documents = reader.load_data(file_path)
    
    parser = MarkdownNodeParser()
    nodes = parser.get_nodes_from_documents(documents)

    # paper_id 삽입
    for node in nodes:
        node.metadata["paper_id"] = paper_id
    
    print(f"논문을 {len(nodes)}개의 노드(Node)로 분할했습니다.")
    return nodes

In [5]:
test_pdf = "./data/paper.pdf"
nodes = load_and_parse_with_docling(test_pdf)

# 첫 번째 노드 확인 (내용과 메타데이터)
print(f"샘플 노드 내용:\n{nodes[0].get_content()[:300]}")
print(f"메타데이터: {nodes[0].metadata}")

[INFO] 2026-03-02 22:26:53,954 [RapidOCR] base.py:22: Using engine_name: onnxruntime
[INFO] 2026-03-02 22:26:53,962 [RapidOCR] download_file.py:60: File exists and is valid: C:\Users\user\miniconda3\envs\rag\Lib\site-packages\rapidocr\models\ch_PP-OCRv4_det_infer.onnx
[INFO] 2026-03-02 22:26:53,964 [RapidOCR] main.py:53: Using C:\Users\user\miniconda3\envs\rag\Lib\site-packages\rapidocr\models\ch_PP-OCRv4_det_infer.onnx
[INFO] 2026-03-02 22:26:54,229 [RapidOCR] base.py:22: Using engine_name: onnxruntime
[INFO] 2026-03-02 22:26:54,237 [RapidOCR] download_file.py:60: File exists and is valid: C:\Users\user\miniconda3\envs\rag\Lib\site-packages\rapidocr\models\ch_ppocr_mobile_v2.0_cls_infer.onnx
[INFO] 2026-03-02 22:26:54,237 [RapidOCR] main.py:53: Using C:\Users\user\miniconda3\envs\rag\Lib\site-packages\rapidocr\models\ch_ppocr_mobile_v2.0_cls_infer.onnx
[INFO] 2026-03-02 22:26:54,448 [RapidOCR] base.py:22: Using engine_name: onnxruntime
[INFO] 2026-03-02 22:26:54,489 [RapidOCR] downloa

논문을 23개의 노드(Node)로 분할했습니다.
샘플 노드 내용:
## VLANeXt: Recipes for Building Strong VLA Models

Xiao-Ming Wu 1 Bin Fan 2 Kang Liao 1 Jian-jian Jiang 2 Runze Yang 3 Yihang Luo 1 Zhonghua Wu 4 Wei-Shi Zheng 2 Chen Change Loy 1 5

https://dravenalg.github.io/VLANeXt/
메타데이터: {'header_path': '/', 'paper_id': '5313b60e59bbcb9baa9d087b3b82d5cb64f52a6e9d819c558fd325713b240d4b'}


# 저장

In [6]:
node_dicts = [node.to_dict() for node in nodes]

with open("./nodes.json", "w", encoding='utf-8') as f:
    json.dump(node_dicts, f, ensure_ascii=False, indent=4)